# Objective: validate the canonical site CSV

Create one fixture row with exactly seven site-level columns. The submission contains capacities, not hourly rows, public generator IDs, storage IDs, or state-of-charge fields.

In [ ]:
from pathlib import Path
import csv, json, os, sys

TUTORIAL_ROOT = Path(os.environ.get('TUTORIAL_ROOT', Path.cwd())).resolve()
OUTPUT_ROOT = Path(os.environ.get('TUTORIAL_OUTPUT_ROOT', TUTORIAL_ROOT / 'outputs')).resolve(); OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIXTURE_ROOT = OUTPUT_ROOT / 'fixtures'; FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)
csv_path = FIXTURE_ROOT / 'student_submission.csv'
columns = ['team_id', 'site_id', 'bus_id', 'connection_capacity_mw', 'generator_capacity_mw', 'storage_power_mw', 'storage_energy_mwh']
row = ['fixture-team', 'site-north', '1001', '40', '20', '5', '20']
with csv_path.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle); writer.writerow(columns); writer.writerow(row)


In [ ]:
print('Environment report')
print({'python': sys.version.split()[0], 'validator': 'ercot_game.csv_validator', 'fixture_first': True})


In [ ]:
assert columns == ['team_id', 'site_id', 'bus_id', 'connection_capacity_mw', 'generator_capacity_mw', 'storage_power_mw', 'storage_energy_mwh']
assert len(columns) == 7
assert float(row[4]) <= float(row[3])
assert float(row[6]) == 4 * float(row[5])
from ercot_game.csv_validator import validate_csv_submission
report = validate_csv_submission(csv_path, bus_ids={'1001', '1002'}, horizon=4368)
assert report.valid, [issue.display() for issue in report.issues]
print('validator valid:', report.valid, 'site_count:', report.site_count, 'horizon label:', report.horizon)


In [ ]:
OUTPUT = OUTPUT_ROOT / 'csv_validation.json'
payload = {'valid': report.valid, 'columns': columns, 'row_count': report.row_count, 'site_count': report.site_count, 'bus_count': report.bus_count, 'horizon_label': 'one-hour half-year scenarios', 'issues': [issue.display() for issue in report.issues]}
OUTPUT.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print('OUTPUT:', OUTPUT)
